# NB11 — ENIGMA FRC Site-Specific Replication

**Purpose:** Test whether per-Mb metal-gene density correlates with local groundwater metal concentrations at the Oak Ridge FRC (Field Research Centre) site. This is a small-scale, site-specific replication of the main hypothesis using culture-independent MAG data.

## Pre-specifications (locked before execution)

| Parameter | Value |
|-----------|-------|
| Metals tested | Cu, Ni, Zn, As, Mn, Cr, Co (groundwater mg/L; ≥90 non-null samples each) |
| Expected direction | **Positive** — higher well-level groundwater metal → higher per-Mb metal-gene density in MAGs from that well |
| Primary unit of analysis | MAG-level (n≈185 MAGs; note: MAGs from the same well share geochemistry — pseudo-replication flagged) |
| Sensitivity | Well-level aggregation (n≤21 wells) |
| Method | Spearman rank correlation (no PGLS — no phylogenetic tree for site-level replication) |
| Multiple testing | Benjamini-Hochberg FDR across 7 metals |
| KO set | Tier 1+2, 140 KOs from `curated_mrg_ko_ids_v2.csv` |
| Genome size | `browser_genome.size` (bp → Mb = size / 1e6) |
| Geochemistry source | `enigma_coral.ddt_brick0000007`; per-well **median** across date-stamped measurements |
| Combined burden | Mean of z-scored per-well concentrations across all 7 metals |
| Exclusion | Uranium excluded (column absent from ddt_brick0000007) |

**Rationale for positive direction:** The FRC is a uranium-contaminated groundwater site with elevated co-contaminant metals (Cu, Ni, Zn, Cr, Co, Mn, As). MAGs from wells with higher metal load should be under stronger selection for metal tolerance/resistance genes, producing a positive correlation between local metal concentration and per-Mb metal-gene density. This is a distinct question from the global P1 finding (β < 0), which shows that genera with higher metal-gene density occupy narrower ecological niches (specialist genera) — here we are testing within-site metal adaptation at the genome level, not global niche breadth.

**Data sources:**
- `enigma_genome_depot_enigma.browser_genome` + `browser_gene` + `browser_protein_kegg_orthologs` + `browser_kegg_ortholog` → MAG KO content
- `enigma_genome_depot_enigma.browser_sample` → well name lookup (browser_sample.id → browser_genome.sample_id)
- `enigma_coral.ddt_brick0000007` → groundwater metal concentrations per well

## Block 0 — Imports and Spark

In [1]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

_project_root = Path('.').resolve()
if _project_root.name == 'notebooks':
    _project_root = _project_root.parent

DATA = _project_root / 'data'

sys.path.insert(0, str(_project_root / 'scripts'))
try:
    from berdl_utils import get_spark_session
    spark = get_spark_session()
except Exception:
    import findspark; findspark.init()
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName('enigma_frc').getOrCreate()

print('Spark version:', spark.version)

[berdl_utils] JupyterHub SparkSession acquired: 4.0.1
Spark version: 4.0.1


## Block 1 — Schema Confirmation

In [2]:
def _describe_table(ns, tbl):
    df = spark.sql(f'DESCRIBE TABLE {ns}.{tbl}').toPandas()
    cols = [r for r in df['col_name'].tolist() if not str(r).startswith('#') and str(r).strip()]
    print(f'--- {ns}.{tbl} ({len(cols)} cols) ---')
    print(', '.join(cols))
    return cols

_describe_table('enigma_genome_depot_enigma', 'browser_genome')
_describe_table('enigma_genome_depot_enigma', 'browser_gene')
_describe_table('enigma_genome_depot_enigma', 'browser_protein_kegg_orthologs')
_describe_table('enigma_genome_depot_enigma', 'browser_kegg_ortholog')
_describe_table('enigma_genome_depot_enigma', 'browser_sample')

--- enigma_genome_depot_enigma.browser_genome (14 cols) ---
id, name, description, contigs, size, genes, json_url, pub_date, external_url, external_id, gbk_filepath, sample_id, strain_id, taxon_id


--- enigma_genome_depot_enigma.browser_gene (12 cols) ---
id, name, locus_tag, type, start, end, strand, function, contig_id, genome_id, operon_id, protein_id


--- enigma_genome_depot_enigma.browser_protein_kegg_orthologs (3 cols) ---
id, protein_id, kegg_ortholog_id


--- enigma_genome_depot_enigma.browser_kegg_ortholog (3 cols) ---
id, kegg_id, description


--- enigma_genome_depot_enigma.browser_sample (4 cols) ---
id, sample_id, full_name, description


['id', 'sample_id', 'full_name', 'description']

In [3]:
# Confirm genome size units: print smallest and largest MAG sizes
r = spark.sql('''
    SELECT g.id, g.name, g.size, g.genes, s.sample_id AS well_id
    FROM enigma_genome_depot_enigma.browser_genome g
    JOIN enigma_genome_depot_enigma.browser_sample s ON s.id = g.sample_id
    WHERE g.sample_id IS NOT NULL
    ORDER BY g.size
    LIMIT 5
''').toPandas()
print('Smallest MAGs (size in bp):')
print(r.to_string(index=False))

r2 = spark.sql('''
    SELECT COUNT(*) AS n_mags, MIN(size) AS min_bp, MAX(size) AS max_bp,
           PERCENTILE_APPROX(size, 0.5) AS median_bp
    FROM enigma_genome_depot_enigma.browser_genome
    WHERE sample_id IS NOT NULL
''').toPandas()
print('\nMAG size summary:')
print(r2.to_string(index=False))

Smallest MAGs (size in bp):
 id         name   size  genes well_id
152 FW215_bin.14 667355    723   FW215
227  FW305_bin.8 708096    764   FW305
156 FW215_bin.29 736572    949   FW215
190 FW300_bin.56 768084    862   FW300
694 GW715_bin.47 787008    874   GW715



MAG size summary:
 n_mags  min_bp  max_bp  median_bp
    185  667355 9392876    3172576


## Block 2 — Load Primary KO Set

In [4]:
gl = pd.read_csv(DATA / 'curated_mrg_ko_ids_v2.csv')
print(gl.columns.tolist())
print(gl.head())

['KO', 'gene_name', 'definition', 'ec', 'primary_category', 'is_resistance', 'is_transport', 'is_sensor', 'is_cofactor', 'is_metabolism', 'overlap_flag', 'pfam_ids', 'pfam_metal', 'evidence_tier', 'tier_1_vs_2', 'metals', 'metal_sources', 'source_kegg_module', 'source_bacmet', 'source_fitness', 'notes']
       KO gene_name                                         definition  \
0  K00013      hisD             histidinol dehydrogenase [EC:1.1.1.23]   
1  K00024       mdh                 malate dehydrogenase [EC:1.1.1.37]   
2  K00031      IDH1             isocitrate dehydrogenase [EC:1.1.1.42]   
3  K00058      serA  D-3-phosphoglycerate dehydrogenase / 2-oxoglut...   
4  K00059      fabG  3-oxoacyl-[acyl-carrier protein] reductase [EC...   

          ec            primary_category  is_resistance  is_transport  \
0   1.1.1.23  Metal-dependent Metabolism          False         False   
1   1.1.1.37  Metal-dependent Metabolism          False         False   
2   1.1.1.42  Metal-dependent M

In [6]:
gl = pd.read_csv(DATA / 'curated_mrg_ko_ids_v2.csv')

# Use .str.startswith to capture 'Tier 1' and any 'Tier 2' variants
primary_kos = set(
    gl.loc[gl['evidence_tier'].str.startswith(('Tier 1', 'Tier 2')), 'KO']
    .str.strip()
)
print(f'Primary KO set: {len(primary_kos)} KOs')
print('Sample KO IDs:', sorted(list(primary_kos))[:5])

Primary KO set: 256 KOs
Sample KO IDs: ['K00013', 'K00058', 'K00059', 'K00108', 'K00127']


## Block 3 — Query MAG KO Densities

Join path: `browser_genome → browser_gene → browser_protein_kegg_orthologs → browser_kegg_ortholog`

Filter to MAGs only (`browser_genome.sample_id IS NOT NULL`). Count distinct primary KOs per MAG.

In [7]:
# Build KO set string for SQL IN clause
# browser_kegg_ortholog.kegg_id format: 'K02313' (no prefix)
# Confirm format by looking at a few entries
r_ko_fmt = spark.sql('''
    SELECT kegg_id, description
    FROM enigma_genome_depot_enigma.browser_kegg_ortholog
    LIMIT 5
''').toPandas()
print('browser_kegg_ortholog sample rows:')
print(r_ko_fmt.to_string(index=False))

browser_kegg_ortholog sample rows:
kegg_id                                        description
 K02313    dnaA; chromosomal replication initiator protein
 K02338 dnaN; DNA polymerase III subunit beta [EC:2.7.7.7]
 K03629      recF; DNA replication and repair protein RecF
 K02470           gyrB; DNA gyrase subunit B [EC:5.99.1.3]
 K03832                     tonB; periplasmic protein TonB


In [8]:
# Adjust KO format based on above output
# If kegg_id is 'K02313', use as-is; if 'ko:K02313', strip prefix
sample_kegg_id = r_ko_fmt['kegg_id'].iloc[0]
if sample_kegg_id.startswith('ko:') or sample_kegg_id.startswith('K0') or sample_kegg_id.startswith('K1'):
    if sample_kegg_id.startswith('ko:'):
        ko_sql_vals = ','.join(f"'ko:{k}'" for k in primary_kos)
    else:
        ko_sql_vals = ','.join(f"'{k}'" for k in primary_kos)
else:
    ko_sql_vals = ','.join(f"'{k}'" for k in primary_kos)

print(f'Format: kegg_id starts with "{sample_kegg_id[:6]}" — using {ko_sql_vals[:60]}...')

Format: kegg_id starts with "K02313" — using 'K03439','K19594','K11704','K02007','K11606','K02225','K0052...


In [9]:
# Main query: MAG KO density
# Note: browser_gene.protein_id → browser_protein_kegg_orthologs.protein_id (both are browser_protein.id FK)
mag_ko_query = f'''
    SELECT
        g.id           AS genome_id,
        g.name         AS genome_name,
        g.size         AS genome_size_bp,
        g.genes        AS n_genes,
        s.sample_id    AS well_id,
        COUNT(DISTINCT ko.kegg_id) AS n_primary_ko
    FROM enigma_genome_depot_enigma.browser_genome g
    JOIN enigma_genome_depot_enigma.browser_sample s
        ON s.id = g.sample_id
    JOIN enigma_genome_depot_enigma.browser_gene gene
        ON gene.genome_id = g.id
    JOIN enigma_genome_depot_enigma.browser_protein_kegg_orthologs pko
        ON pko.protein_id = gene.protein_id
    JOIN enigma_genome_depot_enigma.browser_kegg_ortholog ko
        ON ko.id = pko.kegg_ortholog_id
    WHERE g.sample_id IS NOT NULL
    AND ko.kegg_id IN ({ko_sql_vals})
    GROUP BY g.id, g.name, g.size, g.genes, s.sample_id
'''

mag_ko_df = spark.sql(mag_ko_query).toPandas()
print(f'MAGs with ≥1 primary KO: {len(mag_ko_df)}')
print(f'Wells represented: {mag_ko_df["well_id"].nunique()}')
print(mag_ko_df.describe())

MAGs with ≥1 primary KO: 185
Wells represented: 21
         genome_id  genome_size_bp      n_genes  n_primary_ko
count   185.000000    1.850000e+02   185.000000    185.000000
mean    869.805405    3.289147e+06  3225.205405     71.410811
std    1307.202766    1.453456e+06  1291.269569     27.197224
min      36.000000    6.673550e+05   723.000000      8.000000
25%     121.000000    2.442555e+06  2532.000000     59.000000
50%     192.000000    3.172576e+06  3146.000000     74.000000
75%     690.000000    4.039843e+06  3831.000000     92.000000
max    3784.000000    9.392876e+06  8292.000000    127.000000


In [10]:
# Also get total MAG count (including those with 0 primary KOs)
# to confirm denominator for KO density
all_mags = spark.sql('''
    SELECT g.id AS genome_id, g.name AS genome_name, g.size AS genome_size_bp, s.sample_id AS well_id
    FROM enigma_genome_depot_enigma.browser_genome g
    JOIN enigma_genome_depot_enigma.browser_sample s ON s.id = g.sample_id
    WHERE g.sample_id IS NOT NULL
''').toPandas()

print(f'Total MAGs: {len(all_mags)}')
print(f'MAGs with ≥1 primary KO: {len(mag_ko_df)} ({100*len(mag_ko_df)/len(all_mags):.1f}%)')

# Left join to include MAGs with 0 primary KOs
mag_density = all_mags.merge(mag_ko_df[['genome_id', 'n_primary_ko']],
                              on='genome_id', how='left')
mag_density['n_primary_ko'] = mag_density['n_primary_ko'].fillna(0)
mag_density['genome_size_mb'] = mag_density['genome_size_bp'] / 1e6
mag_density['ko_per_mb'] = mag_density['n_primary_ko'] / mag_density['genome_size_mb']

print(f'\nKO density summary (all {len(mag_density)} MAGs):')
print(mag_density[['n_primary_ko', 'genome_size_mb', 'ko_per_mb']].describe().round(3))

Total MAGs: 185
MAGs with ≥1 primary KO: 185 (100.0%)

KO density summary (all 185 MAGs):
       n_primary_ko  genome_size_mb  ko_per_mb
count       185.000         185.000    185.000
mean         71.411           3.289     22.656
std          27.197           1.453      6.076
min           8.000           0.667      8.607
25%          59.000           2.443     18.538
50%          74.000           3.173     22.402
75%          92.000           4.040     27.538
max         127.000           9.393     37.943


## Block 4 — Fetch and Aggregate Geochemistry per Well

Groundwater metal columns from `enigma_coral.ddt_brick0000007` (mg/L).
Well ID extracted from `sdt_sample_name` via startswith match to `browser_sample.sample_id` values (longest match first, to correctly distinguish FW106 from FW106-02).

In [11]:
# Pre-specified groundwater metal columns (mg/L) with ≥90 non-null rows
METAL_COLS = {
    'Cu': 'concentration_molecule_from_list_copper_atom_milligram_per_liter',
    'Ni': 'concentration_molecule_from_list_nickel_atom_milligram_per_liter',
    'Zn': 'concentration_molecule_from_list_zinc_atom_milligram_per_liter',
    'As': 'concentration_molecule_from_list_arsane_milligram_per_liter',
    'Mn': 'concentration_molecule_from_list_manganese_atom_milligram_per_liter',
    'Cr': 'concentration_molecule_from_list_chromium_atom_milligram_per_liter',
    'Co': 'concentration_molecule_from_list_cobalt_atom_milligram_per_liter',
}

metal_select = ', '.join(f'`{col}` AS {abbrev}' for abbrev, col in METAL_COLS.items())

geo_raw = spark.sql(f'''
    SELECT sdt_sample_name, {metal_select}
    FROM enigma_coral.ddt_brick0000007
    WHERE sdt_sample_name IS NOT NULL
''').toPandas()

print(f'Geochemistry rows: {len(geo_raw)}')
print('Sample names (first 10):', geo_raw['sdt_sample_name'].tolist()[:10])
print('\nNon-null counts per metal:')
print(geo_raw[list(METAL_COLS.keys())].count())

Geochemistry rows: 300
Sample names (first 10): ['GW460-11-04-13', 'GW456-11-04-13', 'GW456-11-04-13-2', 'FW301-11-05-13', 'FW300-11-05-13', 'GW460-11-05-13', 'GW456-11-05-13', 'FW301-11-06-13', 'FW300-11-06-13', 'GW460-11-06-13']

Non-null counts per metal:
Cu    127
Ni    122
Zn    132
As    130
Mn    126
Cr     94
Co     90
dtype: int64


In [12]:
# Match each sdt_sample_name to a browser_sample.sample_id well_id
# Sort well IDs by length descending so 'FW106-02' is tried before 'FW106'
all_well_ids = sorted(mag_density['well_id'].unique().tolist(), key=len, reverse=True)
print(f'Well IDs from MAG database ({len(all_well_ids)}):', all_well_ids)

def _match_well(sample_name, well_ids):
    for w in well_ids:
        if sample_name.startswith(w):
            return w
    return None

geo_raw['well_id'] = geo_raw['sdt_sample_name'].apply(
    lambda x: _match_well(str(x), all_well_ids)
)

matched = geo_raw['well_id'].notna().sum()
print(f'\nGeochemistry rows matched to a MAG well: {matched}/{len(geo_raw)}')
print('Wells with geochemistry data:')
print(geo_raw.loc[geo_raw['well_id'].notna(), 'well_id'].value_counts())

Well IDs from MAG database (21): ['FW106-02', 'FW106-10', 'FW301-02', 'FW306_01', 'FW306_02', 'FW306_03', 'FW306_04', 'FW306_05', 'FW306_06', 'DP16D', 'FW021', 'FW104', 'FW106', 'FW215', 'FW300', 'FW301', 'FW305', 'FW602', 'GW199', 'GW715', 'GW928']

Geochemistry rows matched to a MAG well: 121/300
Wells with geochemistry data:
well_id
FW301       38
FW300       33
FW106-02     9
FW215        8
FW021        8
FW305        7
FW602        7
FW104        7
FW106        3
DP16D        1
Name: count, dtype: int64


In [13]:
# Aggregate geochemistry per well: median across time points
geo_well = (geo_raw
            .dropna(subset=['well_id'])
            .groupby('well_id')[list(METAL_COLS.keys())]
            .median()
            .reset_index())

# Count measurements per well per metal
geo_counts = (geo_raw
              .dropna(subset=['well_id'])
              .groupby('well_id')[list(METAL_COLS.keys())]
              .count()
              .reset_index()
              .rename(columns={m: f'{m}_n' for m in METAL_COLS.keys()}))

geo_well = geo_well.merge(geo_counts, on='well_id')

print(f'Wells with geochemistry: {len(geo_well)}')
print(geo_well[['well_id'] + list(METAL_COLS.keys())].to_string(index=False))

Wells with geochemistry: 10
 well_id       Cu       Ni       Zn        As       Mn       Cr       Co
   DP16D      NaN      NaN      NaN       NaN      NaN      NaN      NaN
   FW021      NaN      NaN      NaN       NaN      NaN      NaN      NaN
   FW104      NaN      NaN      NaN       NaN      NaN      NaN      NaN
   FW106      NaN      NaN      NaN       NaN      NaN      NaN      NaN
FW106-02      NaN      NaN      NaN       NaN      NaN      NaN      NaN
   FW215 0.001400 0.002700 0.002200 14.170000 0.120000 0.002100      NaN
   FW300 0.002757 0.002817 0.047056  0.002035 0.017214 0.001834 0.001901
   FW301 0.003536 0.005213 0.037826  0.001541 0.014519 0.006216 0.003145
   FW305      NaN      NaN      NaN       NaN      NaN      NaN      NaN
   FW602      NaN      NaN      NaN       NaN      NaN      NaN      NaN


## Block 5 — Join MAG Density to Geochemistry

In [14]:
# MAG-level join: each MAG gets its well's geochemistry values
mag_geo = mag_density.merge(geo_well, on='well_id', how='inner')

print(f'MAGs with geochemistry: {len(mag_geo)}')
print(f'Wells in joined dataset: {mag_geo["well_id"].nunique()}')
print(f'MAGs not matched (dropped): {len(mag_density) - len(mag_geo)}')

# Summary by well
well_summary = (mag_geo.groupby('well_id')
                .agg(n_mags=('genome_id', 'count'),
                     mean_ko_per_mb=('ko_per_mb', 'mean'),
                     median_ko_per_mb=('ko_per_mb', 'median'))
                .reset_index())

print('\nMAGs per well:')
print(well_summary.sort_values('n_mags', ascending=False).to_string(index=False))

MAGs with geochemistry: 109
Wells in joined dataset: 10
MAGs not matched (dropped): 76

MAGs per well:
 well_id  n_mags  mean_ko_per_mb  median_ko_per_mb
   FW021      24       24.022283         23.998911
   FW602      18       21.731206         22.128318
   FW300      16       22.770866         21.012342
   DP16D      15       23.755250         23.254259
   FW215      10       18.500160         18.633182
FW106-02       9       28.463598         28.998517
   FW305       6       23.303871         23.524825
   FW106       5       24.804141         24.718602
   FW104       3       28.707110         29.057682
   FW301       3       18.003890         19.726295


## Block 6 — Spearman Correlations (MAG-level, n≈185)

**Pre-specified direction:** positive rho (higher metal → higher density). One-tailed p-values reported alongside two-tailed for transparency. BH correction on two-tailed p-values.

In [15]:
def spearman_row(data, metal, density_col='ko_per_mb', direction='positive'):
    sub = data[[density_col, metal]].dropna()
    n = len(sub)
    if n < 5:
        return {'metal': metal, 'n_mags': n, 'rho': np.nan,
                'p_two_tailed': np.nan, 'p_one_tailed': np.nan, 'note': 'n<5'}
    rho, p2 = stats.spearmanr(sub[density_col], sub[metal])
    # One-tailed p: if expected direction is positive, p_one = p_two/2 when rho>0
    if direction == 'positive':
        p1 = p2 / 2 if rho > 0 else 1 - p2 / 2
    else:
        p1 = p2 / 2 if rho < 0 else 1 - p2 / 2
    return {'metal': metal, 'n_mags': n, 'rho': round(rho, 4),
            'p_two_tailed': round(p2, 4), 'p_one_tailed': round(p1, 4), 'note': ''}

mag_results = []
for metal in METAL_COLS:
    row = spearman_row(mag_geo, metal)
    mag_results.append(row)

mag_corr = pd.DataFrame(mag_results)

# BH correction on two-tailed p-values
valid_mask = mag_corr['p_two_tailed'].notna()
if valid_mask.sum() > 0:
    rej, p_adj, _, _ = multipletests(mag_corr.loc[valid_mask, 'p_two_tailed'], method='fdr_bh')
    mag_corr.loc[valid_mask, 'p_fdr'] = p_adj.round(4)
    mag_corr.loc[valid_mask, 'significant'] = rej

print('MAG-level Spearman correlations (n≈185 MAGs, pseudo-replication within wells):')
print(mag_corr.to_string(index=False))

MAG-level Spearman correlations (n≈185 MAGs, pseudo-replication within wells):
metal  n_mags     rho  p_two_tailed  p_one_tailed note  p_fdr significant
   Cu      29  0.1659        0.3897        0.1949      0.3897       False
   Ni      29  0.1659        0.3897        0.1949      0.3897       False
   Zn      29  0.3803        0.0419        0.0209      0.1466       False
   As      29 -0.1659        0.3897        0.8051      0.3897       False
   Mn      29 -0.1659        0.3897        0.8051      0.3897       False
   Cr      29 -0.4069        0.0285        0.9858      0.1466       False
   Co      19 -0.3689        0.1201        0.9400      0.2802       False


## Block 7 — Well-Level Sensitivity (n≤21 wells)

Aggregate ko_per_mb to median per well before correlating. This removes within-well pseudo-replication; n is small but each point is independent.

In [16]:
# Well-level dataset: median KO density per well
well_density = (mag_geo.groupby('well_id')
                .agg(n_mags=('genome_id', 'count'),
                     median_ko_per_mb=('ko_per_mb', 'median'),
                     mean_ko_per_mb=('ko_per_mb', 'mean'))
                .reset_index())
well_geo = well_density.merge(geo_well, on='well_id', how='inner')

print(f'Well-level dataset: {len(well_geo)} wells')
print(well_geo[['well_id', 'n_mags', 'median_ko_per_mb'] + list(METAL_COLS.keys())].to_string(index=False))

Well-level dataset: 10 wells
 well_id  n_mags  median_ko_per_mb       Cu       Ni       Zn        As       Mn       Cr       Co
   DP16D      15         23.254259      NaN      NaN      NaN       NaN      NaN      NaN      NaN
   FW021      24         23.998911      NaN      NaN      NaN       NaN      NaN      NaN      NaN
   FW104       3         29.057682      NaN      NaN      NaN       NaN      NaN      NaN      NaN
   FW106       5         24.718602      NaN      NaN      NaN       NaN      NaN      NaN      NaN
FW106-02       9         28.998517      NaN      NaN      NaN       NaN      NaN      NaN      NaN
   FW215      10         18.633182 0.001400 0.002700 0.002200 14.170000 0.120000 0.002100      NaN
   FW300      16         21.012342 0.002757 0.002817 0.047056  0.002035 0.017214 0.001834 0.001901
   FW301       3         19.726295 0.003536 0.005213 0.037826  0.001541 0.014519 0.006216 0.003145
   FW305       6         23.524825      NaN      NaN      NaN       NaN      NaN

In [17]:
well_results = []
for metal in METAL_COLS:
    row = spearman_row(well_geo, metal, density_col='median_ko_per_mb')
    row['level'] = 'well'
    well_results.append(row)

well_corr = pd.DataFrame(well_results)

valid_mask_w = well_corr['p_two_tailed'].notna()
if valid_mask_w.sum() > 0:
    rej_w, p_adj_w, _, _ = multipletests(well_corr.loc[valid_mask_w, 'p_two_tailed'], method='fdr_bh')
    well_corr.loc[valid_mask_w, 'p_fdr'] = p_adj_w.round(4)
    well_corr.loc[valid_mask_w, 'significant'] = rej_w

print('Well-level Spearman correlations (median KO density per well):')
print(well_corr.to_string(index=False))

Well-level Spearman correlations (median KO density per well):
metal  n_mags  rho  p_two_tailed  p_one_tailed note level
   Cu       3  NaN           NaN           NaN  n<5  well
   Ni       3  NaN           NaN           NaN  n<5  well
   Zn       3  NaN           NaN           NaN  n<5  well
   As       3  NaN           NaN           NaN  n<5  well
   Mn       3  NaN           NaN           NaN  n<5  well
   Cr       3  NaN           NaN           NaN  n<5  well
   Co       2  NaN           NaN           NaN  n<5  well


## Block 8 — Combined Metal Burden Score

In [22]:
# Combined metal burden: mean of z-scored per-well concentrations across all 7 metals
metals = list(METAL_COLS.keys())

geo_z = geo_well.copy()
for m in metals:
    col_vals = geo_z[m]
    if col_vals.std() > 0:
        geo_z[f'{m}_z'] = (col_vals - col_vals.mean()) / col_vals.std()
    else:
        geo_z[f'{m}_z'] = 0.0

z_cols = [f'{m}_z' for m in metals]
geo_z['metal_burden'] = geo_z[z_cols].mean(axis=1)

print("geo_z shape:", geo_z.shape)
print("metal_burden std:", geo_z['metal_burden'].std())
print("metal_burden range:", geo_z['metal_burden'].min(), geo_z['metal_burden'].max())

# --- MAG-level burden ---
mag_burden = mag_geo.merge(geo_z[['well_id', 'metal_burden']], on='well_id', how='inner')
print(f"MAG-level merged: {len(mag_burden)} rows")

# Only keep rows where both variables are non-null
mag_valid = mag_burden.dropna(subset=['ko_per_mb', 'metal_burden'])
print(f"  Valid pairs (both non-null): {len(mag_valid)}")
if len(mag_valid) > 2:
    print(f"  ko_per_mb std: {mag_valid['ko_per_mb'].std():.6f}")
    print(f"  metal_burden std: {mag_valid['metal_burden'].std():.6f}")

if len(mag_valid) > 2 and mag_valid['ko_per_mb'].std() > 1e-10 and mag_valid['metal_burden'].std() > 1e-10:
    rho_b, p_b = stats.spearmanr(mag_valid['ko_per_mb'], mag_valid['metal_burden'])
    n_b = len(mag_valid)
    print(f'Combined metal burden ~ ko_per_mb (MAG-level, n={n_b}): rho={rho_b:.4f}, p={p_b:.4f}')
else:
    rho_b, p_b = np.nan, np.nan
    n_b = len(mag_valid)
    print(f'MAG-level: insufficient valid pairs or no variation. n={n_b}')

# --- Well-level burden ---
well_burden = well_density.merge(geo_z[['well_id', 'metal_burden']], on='well_id', how='inner')
print(f"Well-level merged: {len(well_burden)} rows")
well_valid = well_burden.dropna(subset=['median_ko_per_mb', 'metal_burden'])
print(f"  Valid pairs (both non-null): {len(well_valid)}")
if len(well_valid) > 2:
    print(f"  median_ko_per_mb std: {well_valid['median_ko_per_mb'].std():.6f}")
    print(f"  metal_burden std: {well_valid['metal_burden'].std():.6f}")

if len(well_valid) > 2 and well_valid['median_ko_per_mb'].std() > 1e-10 and well_valid['metal_burden'].std() > 1e-10:
    rho_ww, p_ww = stats.spearmanr(well_valid['median_ko_per_mb'], well_valid['metal_burden'])
    n_ww = len(well_valid)
    print(f'Combined metal burden ~ median_ko_per_mb (well-level, n={n_ww}): rho={rho_ww:.4f}, p={p_ww:.4f}')
else:
    rho_ww, p_ww = np.nan, np.nan
    n_ww = len(well_valid)
    print(f'Well-level: insufficient valid pairs or no variation. n={n_ww}')

burden_results = [
    {'metal': 'combined_burden', 'level': 'MAG', 'n': n_b, 'rho': rho_b, 'p_two_tailed': p_b},
    {'metal': 'combined_burden', 'level': 'well', 'n': n_ww, 'rho': rho_ww, 'p_two_tailed': p_ww},
]

print("\nBurden results:")
print(pd.DataFrame(burden_results))

geo_z shape: (10, 23)
metal_burden std: 0.39623902080329454
metal_burden range: -0.2951175240700198 0.4438567919459719
MAG-level merged: 109 rows
  Valid pairs (both non-null): 29
  ko_per_mb std: 5.360923
  metal_burden std: 0.221983
Combined metal burden ~ ko_per_mb (MAG-level, n=29): rho=-0.4069, p=0.0285
Well-level merged: 10 rows
  Valid pairs (both non-null): 3
  median_ko_per_mb std: 1.190883
  metal_burden std: 0.396239
Combined metal burden ~ median_ko_per_mb (well-level, n=3): rho=-0.5000, p=0.6667

Burden results:
             metal level   n       rho  p_two_tailed
0  combined_burden   MAG  29 -0.406854      0.028499
1  combined_burden  well   3 -0.500000      0.666667


## Block 9 — Save Results

In [27]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

# 1. Define metal columns (adjust to your actual column names if different)
METAL_COLS = {
    'Cu': 'Cu_ppm',
    'Ni': 'Ni_ppm',
    'Zn': 'Zn_ppm',
    'As': 'As_ppm',
    'Mn': 'Mn_ppm',
    'Cr': 'Cr_ppm',
    'Co': 'Co_ppm',
}
metals = list(METAL_COLS.keys())

# 2. Ensure we have a well-level density DataFrame
try:
    well_density  # check if it exists
except NameError:
    print("Creating well_density from mag_geo (median KO per MB per well)")
    well_density = mag_geo.groupby('well_id').agg(
        median_ko_per_mb=('ko_per_mb', 'median')
    ).reset_index()
    print(f"well_density shape: {well_density.shape}")

# 3. Combined metal burden (mean of z-scored concentrations)
geo_z = geo_well.copy()
for m in metals:
    col_vals = geo_z[m]
    if col_vals.std() > 0:
        geo_z[f'{m}_z'] = (col_vals - col_vals.mean()) / col_vals.std()
    else:
        geo_z[f'{m}_z'] = 0.0

z_cols = [f'{m}_z' for m in metals]
geo_z['metal_burden'] = geo_z[z_cols].mean(axis=1)

print("geo_z shape:", geo_z.shape)
print("metal_burden std:", geo_z['metal_burden'].std())
print("metal_burden range:", geo_z['metal_burden'].min(), geo_z['metal_burden'].max())

# 4. MAG‑level: Spearman between ko_per_mb and combined burden
mag_burden = mag_geo.merge(geo_z[['well_id', 'metal_burden']], on='well_id', how='inner')
mag_valid = mag_burden.dropna(subset=['ko_per_mb', 'metal_burden'])
n_mag = len(mag_valid)
print(f"\nMAG-level valid pairs (ko_per_mb + metal_burden): {n_mag}")

if n_mag > 2 and mag_valid['ko_per_mb'].std() > 1e-10 and mag_valid['metal_burden'].std() > 1e-10:
    rho_b, p_b = stats.spearmanr(mag_valid['ko_per_mb'], mag_valid['metal_burden'])
    print(f'Combined burden ~ ko_per_mb (MAG-level, n={n_mag}): rho={rho_b:.4f}, p={p_b:.4f}')
else:
    rho_b, p_b = np.nan, np.nan
    print(f'MAG-level: insufficient valid pairs or no variation. n={n_mag}')

burden_results = [
    {'metal': 'combined_burden', 'level': 'MAG', 'n': n_mag, 'rho': rho_b, 'p_two_tailed': p_b}
]

# 5. Well‑level: per‑metal correlations and combined burden
# 5a. Per‑metal well‑level correlations
well_corr_list = []

for m in metals:
    # Merge well density with geochemistry for this metal
    merged = well_density.merge(geo_well[['well_id', m]], on='well_id', how='inner')
    valid = merged.dropna(subset=['median_ko_per_mb', m])
    n = len(valid)
    if n > 2 and valid[m].std() > 1e-10 and valid['median_ko_per_mb'].std() > 1e-10:
        rho, p_two = stats.spearmanr(valid['median_ko_per_mb'], valid[m])
    else:
        rho, p_two = np.nan, np.nan
    well_corr_list.append({
        'metal': m,
        'n_mags': None,           # well-level has no MAG count
        'rho': rho,
        'p_two_tailed': p_two,
        'p_one_tailed': p_two / 2 if not np.isnan(p_two) else np.nan,
        'note': '',
        'level': 'well',
        'n_wells': n
    })

# 5b. Well‑level combined burden
well_burden = well_density.merge(geo_z[['well_id', 'metal_burden']], on='well_id', how='inner')
well_valid = well_burden.dropna(subset=['median_ko_per_mb', 'metal_burden'])
n_well = len(well_valid)
print(f"\nWell-level valid pairs (median_ko_per_mb + metal_burden): {n_well}")

if n_well > 2 and well_valid['median_ko_per_mb'].std() > 1e-10 and well_valid['metal_burden'].std() > 1e-10:
    rho_ww, p_ww = stats.spearmanr(well_valid['median_ko_per_mb'], well_valid['metal_burden'])
    print(f'Combined burden ~ median_ko_per_mb (well-level, n={n_well}): rho={rho_ww:.4f}, p={p_ww:.4f}')
else:
    rho_ww, p_ww = np.nan, np.nan
    print(f'Well-level combined burden: insufficient data. n={n_well}')

burden_results.append({
    'metal': 'combined_burden',
    'level': 'well',
    'n': n_well,
    'rho': rho_ww,
    'p_two_tailed': p_ww
})

# Convert to DataFrame
well_corr = pd.DataFrame(well_corr_list)

# 6. Add FDR correction to well_corr (and mag_corr if missing)
def add_fdr(df, p_col='p_two_tailed', fdr_col='p_fdr', sig_col='significant'):
    if fdr_col in df.columns and sig_col in df.columns:
        return df
    pvals = df[p_col].dropna().values
    if len(pvals) > 1:
        reject, p_adj, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')
        p_adj_full = np.full(len(df), np.nan)
        sig_full = np.full(len(df), False, dtype=bool)
        valid_idx = df[p_col].notna()
        p_adj_full[valid_idx] = p_adj
        sig_full[valid_idx] = reject
        df[fdr_col] = p_adj_full
        df[sig_col] = sig_full
    else:
        df[fdr_col] = df[p_col]
        df[sig_col] = df[p_col] < 0.05
    return df

well_corr = add_fdr(well_corr)
mag_corr = add_fdr(mag_corr)  # safe to call even if already present

# 7. Compile full results table
# Add columns to mag_corr and well_corr for concatenation
mag_corr['level'] = 'MAG'
mag_corr['n_wells'] = mag_geo['well_id'].nunique()
well_corr['n_wells'] = len(geo_well)
well_corr['n_mags'] = None

# Define columns to keep
cols = ['metal', 'level', 'n_mags', 'rho', 'p_two_tailed', 'p_one_tailed', 'p_fdr', 'significant']

# Concatenate
results_out = pd.concat([
    mag_corr[cols],
    well_corr[cols],
], ignore_index=True)

burden_df = pd.DataFrame(burden_results)

# 8. Save outputs
results_out.to_csv(DATA / 'enigma_frc_replication.csv', index=False)
mag_geo.to_csv(DATA / 'enigma_frc_mag_geo_joined.csv', index=False)
geo_well.to_csv(DATA / 'enigma_frc_well_geochemistry.csv', index=False)
burden_df.to_csv(DATA / 'enigma_frc_burden_correlations.csv', index=False)

print('\nSaved:')
print('  data/enigma_frc_replication.csv')
print('  data/enigma_frc_mag_geo_joined.csv')
print('  data/enigma_frc_well_geochemistry.csv')
print('  data/enigma_frc_burden_correlations.csv')

# 9. Final summary
print('\n=== FINAL RESULTS SUMMARY ===')
print('\nMAG-level (pseudo-replicated):')
print(mag_corr[['metal', 'n_mags', 'rho', 'p_two_tailed', 'p_fdr', 'significant']].to_string(index=False))

print('\nWell-level (independent):')
print(well_corr[['metal', 'n_mags', 'rho', 'p_two_tailed', 'p_fdr', 'significant']].to_string(index=False))

print('\nCombined burden correlations:')
print(burden_df.to_string(index=False))

geo_z shape: (10, 23)
metal_burden std: 0.39623902080329454
metal_burden range: -0.2951175240700198 0.4438567919459719

MAG-level valid pairs (ko_per_mb + metal_burden): 29
Combined burden ~ ko_per_mb (MAG-level, n=29): rho=-0.4069, p=0.0285

Well-level valid pairs (median_ko_per_mb + metal_burden): 3
Combined burden ~ median_ko_per_mb (well-level, n=3): rho=-0.5000, p=0.6667

Saved:
  data/enigma_frc_replication.csv
  data/enigma_frc_mag_geo_joined.csv
  data/enigma_frc_well_geochemistry.csv
  data/enigma_frc_burden_correlations.csv

=== FINAL RESULTS SUMMARY ===

MAG-level (pseudo-replicated):
metal  n_mags     rho  p_two_tailed  p_fdr significant
   Cu      29  0.1659        0.3897 0.3897       False
   Ni      29  0.1659        0.3897 0.3897       False
   Zn      29  0.3803        0.0419 0.1466       False
   As      29 -0.1659        0.3897 0.3897       False
   Mn      29 -0.1659        0.3897 0.3897       False
   Cr      29 -0.4069        0.0285 0.1466       False
   Co      1

## Block 10 — Interpretation

**Pre-specified decision rule:** If the majority of metals (≥4/7) show rho > 0 AND at least one reaches FDR p < 0.05 (MAG-level) or p < 0.05 uncorrected (well-level, given small n), classify as "Partially supported".

**Caveat:** The FRC site is primarily a uranium/heavy-metal contaminated *groundwater* system. MAGs represent organisms adapted to groundwater conditions, not soil. The geochemistry reflects contamination gradients across wells, not background soil concentrations. This makes the expected positive correlation mechanistically plausible (higher metal load → stronger selection for resistance genes) but not certain (cross-well variation may reflect hydrology as much as metal toxicity).

In [28]:
# Automated interpretation
pos_rho = (mag_corr['rho'] > 0).sum()
sig_fdr = (mag_corr['significant'] == True).sum() if 'significant' in mag_corr.columns else 0
n_metals = len(METAL_COLS)

print(f'Pre-specified metals: {n_metals}')
print(f'Metals with rho > 0 (MAG-level): {pos_rho}/{n_metals}')
print(f'Metals with FDR p < 0.05 (MAG-level): {sig_fdr}/{n_metals}')

if pos_rho >= 4 and sig_fdr >= 1:
    verdict = 'PARTIALLY SUPPORTED — majority positive rho, ≥1 FDR-significant'
elif pos_rho >= 4:
    verdict = 'DIRECTIONALLY CONSISTENT (not significant) — majority positive rho, none FDR-significant'
elif pos_rho <= 2:
    verdict = 'NOT SUPPORTED — majority negative or null rho'
else:
    verdict = 'MIXED — near-equal positive and negative rho'

print(f'\nVerdict: {verdict}')
print('\nUpdate INTERPRETATION_TABLE.md §7 with these values.')

Pre-specified metals: 7
Metals with rho > 0 (MAG-level): 3/7
Metals with FDR p < 0.05 (MAG-level): 0/7

Verdict: MIXED — near-equal positive and negative rho

Update INTERPRETATION_TABLE.md §7 with these values.
